In [1]:
import pandas as pd
import numpy as np

print("Feature engineering notebook ready!")

Feature engineering notebook ready!


In [2]:
# Load raw datasets
train = pd.read_csv(
    "../data/raw/train.csv",
    dtype={"StateHoliday": str}
)

store = pd.read_csv("../data/raw/store.csv")

print("Train shape:", train.shape)
print("Store shape:", store.shape)

Train shape: (1017209, 9)
Store shape: (1115, 10)


In [3]:
# Basic cleaning

# Convert Date to datetime
train["Date"] = pd.to_datetime(train["Date"])

# Standardize StateHoliday
train["StateHoliday"] = train["StateHoliday"].astype(str)

# Fill missing competition distance with median
store["CompetitionDistance"] = store["CompetitionDistance"].fillna(
    store["CompetitionDistance"].median()
)

print("Date dtype:", train["Date"].dtype)
print("Missing CompetitionDistance:",
      store["CompetitionDistance"].isna().sum())

Date dtype: datetime64[us]
Missing CompetitionDistance: 0


In [4]:
df = train.merge(
    store,
    on="Store",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", df.shape)

Merged shape: (1017209, 18)


In [5]:
# Calendar features

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["Quarter"] = df["Date"].dt.quarter
df["DayOfYear"] = df["Date"].dt.dayofyear

df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

print(df[
    [
        "Date",
        "Year",
        "Month",
        "Day",
        "WeekOfYear",
        "Quarter",
        "DayOfYear",
        "IsMonthStart",
        "IsMonthEnd"
    ]
].head())

        Date  Year  Month  Day  WeekOfYear  Quarter  DayOfYear  IsMonthStart  \
0 2015-07-31  2015      7   31          31        3        212             0   
1 2015-07-31  2015      7   31          31        3        212             0   
2 2015-07-31  2015      7   31          31        3        212             0   
3 2015-07-31  2015      7   31          31        3        212             0   
4 2015-07-31  2015      7   31          31        3        212             0   

   IsMonthEnd  
0           1  
1           1  
2           1  
3           1  
4           1  


In [6]:
# Create competition start date

competition_year = df["CompetitionOpenSinceYear"].fillna(
    df["Date"].dt.year
).astype(int)

competition_month = df["CompetitionOpenSinceMonth"].fillna(
    df["Date"].dt.month
).astype(int)

df["CompetitionOpenDate"] = pd.to_datetime(
    {
        "year": competition_year,
        "month": competition_month,
        "day": 1
    }
)

df["CompetitionAgeMonths"] = (
    (df["Date"].dt.year - df["CompetitionOpenDate"].dt.year) * 12
    + (df["Date"].dt.month - df["CompetitionOpenDate"].dt.month)
)

df["CompetitionAgeMonths"] = df["CompetitionAgeMonths"].clip(lower=0)

print(
    df[
        [
            "Date",
            "CompetitionDistance",
            "CompetitionOpenDate",
            "CompetitionAgeMonths"
        ]
    ].head()
)

        Date  CompetitionDistance CompetitionOpenDate  CompetitionAgeMonths
0 2015-07-31               1270.0          2008-09-01                    82
1 2015-07-31                570.0          2007-11-01                    92
2 2015-07-31              14130.0          2006-12-01                   103
3 2015-07-31                620.0          2009-09-01                    70
4 2015-07-31              29910.0          2015-04-01                     3


In [7]:
# Flag unknown competition opening date
df["CompetitionDateMissing"] = (
    df["CompetitionOpenSinceYear"].isna()
    | df["CompetitionOpenSinceMonth"].isna()
).astype(int)

print(df["CompetitionDateMissing"].value_counts())

CompetitionDateMissing
0    693861
1    323348
Name: count, dtype: int64


In [8]:
# Promo2 start date from ISO year and week

promo_year = df["Promo2SinceYear"].fillna(2000).astype(int)
promo_week = df["Promo2SinceWeek"].fillna(1).astype(int)

promo_start_strings = (
    promo_year.astype(str)
    + "-W"
    + promo_week.astype(str).str.zfill(2)
    + "-1"
)

df["Promo2StartDate"] = pd.to_datetime(
    promo_start_strings,
    format="%G-W%V-%u",
    errors="coerce"
)

df["Promo2Started"] = (
    (df["Promo2"] == 1)
    & (df["Date"] >= df["Promo2StartDate"])
).astype(int)

print(
    df[
        [
            "Date",
            "Promo2",
            "Promo2SinceWeek",
            "Promo2SinceYear",
            "Promo2StartDate",
            "Promo2Started"
        ]
    ].head(10)
)

        Date  Promo2  Promo2SinceWeek  Promo2SinceYear Promo2StartDate  \
0 2015-07-31       0              NaN              NaN      2000-01-03   
1 2015-07-31       1             13.0           2010.0      2010-03-29   
2 2015-07-31       1             14.0           2011.0      2011-04-04   
3 2015-07-31       0              NaN              NaN      2000-01-03   
4 2015-07-31       0              NaN              NaN      2000-01-03   
5 2015-07-31       0              NaN              NaN      2000-01-03   
6 2015-07-31       0              NaN              NaN      2000-01-03   
7 2015-07-31       0              NaN              NaN      2000-01-03   
8 2015-07-31       0              NaN              NaN      2000-01-03   
9 2015-07-31       0              NaN              NaN      2000-01-03   

   Promo2Started  
0              0  
1              1  
2              1  
3              0  
4              0  
5              0  
6              0  
7              0  
8             

In [9]:
# Map month number to Rossmann PromoInterval abbreviations
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec"
}

df["MonthName"] = df["Month"].map(month_map)

# Promo2 is active only if:
# 1. Store participates in Promo2
# 2. Promo2 has already started
# 3. Current month is listed in PromoInterval

df["Promo2Active"] = df.apply(
    lambda row: int(
        row["Promo2"] == 1
        and row["Promo2Started"] == 1
        and pd.notna(row["PromoInterval"])
        and row["MonthName"] in row["PromoInterval"].split(",")
    ),
    axis=1
)

print(df["Promo2Active"].value_counts())

print(
    df[
        [
            "Date",
            "Store",
            "MonthName",
            "Promo2",
            "Promo2Started",
            "PromoInterval",
            "Promo2Active"
        ]
    ].head(20)
)

Promo2Active
0    865157
1    152052
Name: count, dtype: int64
         Date  Store MonthName  Promo2  Promo2Started     PromoInterval  \
0  2015-07-31      1       Jul       0              0               NaN   
1  2015-07-31      2       Jul       1              1   Jan,Apr,Jul,Oct   
2  2015-07-31      3       Jul       1              1   Jan,Apr,Jul,Oct   
3  2015-07-31      4       Jul       0              0               NaN   
4  2015-07-31      5       Jul       0              0               NaN   
5  2015-07-31      6       Jul       0              0               NaN   
6  2015-07-31      7       Jul       0              0               NaN   
7  2015-07-31      8       Jul       0              0               NaN   
8  2015-07-31      9       Jul       0              0               NaN   
9  2015-07-31     10       Jul       0              0               NaN   
10 2015-07-31     11       Jul       1              1   Jan,Apr,Jul,Oct   
11 2015-07-31     12       Jul       

In [10]:
# Sort chronologically within each store

df = df.sort_values(
    ["Store", "Date"]
).reset_index(drop=True)

print(
    df[
        ["Store", "Date", "Sales"]
    ].head(10)
)

   Store       Date  Sales
0      1 2013-01-01      0
1      1 2013-01-02   5530
2      1 2013-01-03   4327
3      1 2013-01-04   4486
4      1 2013-01-05   4997
5      1 2013-01-06      0
6      1 2013-01-07   7176
7      1 2013-01-08   5580
8      1 2013-01-09   5471
9      1 2013-01-10   4892


In [11]:
# Create historical sales lag features

lag_days = [1, 7, 14, 28]

for lag in lag_days:
    df[f"SalesLag{lag}"] = (
        df.groupby("Store")["Sales"]
          .shift(lag)
    )

print(
    df[
        [
            "Store",
            "Date",
            "Sales",
            "SalesLag1",
            "SalesLag7",
            "SalesLag14",
            "SalesLag28"
        ]
    ].head(35)
)

    Store       Date  Sales  SalesLag1  SalesLag7  SalesLag14  SalesLag28
0       1 2013-01-01      0        NaN        NaN         NaN         NaN
1       1 2013-01-02   5530        0.0        NaN         NaN         NaN
2       1 2013-01-03   4327     5530.0        NaN         NaN         NaN
3       1 2013-01-04   4486     4327.0        NaN         NaN         NaN
4       1 2013-01-05   4997     4486.0        NaN         NaN         NaN
5       1 2013-01-06      0     4997.0        NaN         NaN         NaN
6       1 2013-01-07   7176        0.0        NaN         NaN         NaN
7       1 2013-01-08   5580     7176.0        0.0         NaN         NaN
8       1 2013-01-09   5471     5580.0     5530.0         NaN         NaN
9       1 2013-01-10   4892     5471.0     4327.0         NaN         NaN
10      1 2013-01-11   4881     4892.0     4486.0         NaN         NaN
11      1 2013-01-12   4952     4881.0     4997.0         NaN         NaN
12      1 2013-01-13      0     4952.0

In [12]:
# Create leakage-safe rolling sales features

df["SalesRollingMean7"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(window=7).mean())
)

df["SalesRollingMean28"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(window=28).mean())
)

df["SalesRollingStd7"] = (
    df.groupby("Store")["Sales"]
      .transform(lambda x: x.shift(1).rolling(window=7).std())
)

print(
    df[
        [
            "Store",
            "Date",
            "Sales",
            "SalesLag1",
            "SalesRollingMean7",
            "SalesRollingMean28",
            "SalesRollingStd7"
        ]
    ].head(35)
)

    Store       Date  Sales  SalesLag1  SalesRollingMean7  SalesRollingMean28  \
0       1 2013-01-01      0        NaN                NaN                 NaN   
1       1 2013-01-02   5530        0.0                NaN                 NaN   
2       1 2013-01-03   4327     5530.0                NaN                 NaN   
3       1 2013-01-04   4486     4327.0                NaN                 NaN   
4       1 2013-01-05   4997     4486.0                NaN                 NaN   
5       1 2013-01-06      0     4997.0                NaN                 NaN   
6       1 2013-01-07   7176        0.0                NaN                 NaN   
7       1 2013-01-08   5580     7176.0        3788.000000                 NaN   
8       1 2013-01-09   5471     5580.0        4585.142857                 NaN   
9       1 2013-01-10   4892     5471.0        4576.714286                 NaN   
10      1 2013-01-11   4881     4892.0        4657.428571                 NaN   
11      1 2013-01-12   4952 

In [13]:
# Demand momentum features

# Short-term vs medium-term demand
df["SalesMomentum"] = (
    df["SalesRollingMean7"] / (df["SalesRollingMean28"] + 1)
)

# Difference between recent demand and monthly demand
df["SalesTrend"] = (
    df["SalesRollingMean7"] - df["SalesRollingMean28"]
)

# Sales change compared with previous week
df["SalesChange7"] = (
    df["SalesLag1"] - df["SalesLag7"]
)

print(
    df[
        [
            "Store",
            "Date",
            "Sales",
            "SalesRollingMean7",
            "SalesRollingMean28",
            "SalesMomentum",
            "SalesTrend",
            "SalesChange7"
        ]
    ].head(35)
)

    Store       Date  Sales  SalesRollingMean7  SalesRollingMean28  \
0       1 2013-01-01      0                NaN                 NaN   
1       1 2013-01-02   5530                NaN                 NaN   
2       1 2013-01-03   4327                NaN                 NaN   
3       1 2013-01-04   4486                NaN                 NaN   
4       1 2013-01-05   4997                NaN                 NaN   
5       1 2013-01-06      0                NaN                 NaN   
6       1 2013-01-07   7176                NaN                 NaN   
7       1 2013-01-08   5580        3788.000000                 NaN   
8       1 2013-01-09   5471        4585.142857                 NaN   
9       1 2013-01-10   4892        4576.714286                 NaN   
10      1 2013-01-11   4881        4657.428571                 NaN   
11      1 2013-01-12   4952        4713.857143                 NaN   
12      1 2013-01-13      0        4707.428571                 NaN   
13      1 2013-01-14

In [14]:
# Audit dataset after feature engineering

print("Dataset shape:", df.shape)

print("\nMissing values:")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\nColumn data types:")
print(df.dtypes)

print("\nTotal columns:", len(df.columns))

Dataset shape: (1017209, 43)

Missing values:
Promo2SinceWeek              508031
PromoInterval                508031
Promo2SinceYear              508031
CompetitionOpenSinceYear     323348
CompetitionOpenSinceMonth    323348
SalesRollingMean28            31220
SalesLag28                    31220
SalesTrend                    31220
SalesMomentum                 31220
SalesLag14                    15610
SalesLag7                      7805
SalesRollingStd7               7805
SalesRollingMean7              7805
SalesChange7                   7805
SalesLag1                      1115
dtype: int64

Column data types:
Store                                 int64
DayOfWeek                             int64
Date                         datetime64[us]
Sales                                 int64
Customers                             int64
Open                                  int64
Promo                                 int64
StateHoliday                            str
SchoolHoliday                

In [15]:
# Keep rows with sufficient historical information

history_features = [
    "SalesLag1",
    "SalesLag7",
    "SalesLag14",
    "SalesLag28",
    "SalesRollingMean7",
    "SalesRollingMean28",
    "SalesRollingStd7",
    "SalesMomentum",
    "SalesTrend",
    "SalesChange7"
]

print("Rows before:", len(df))

df_model = df.dropna(
    subset=history_features
).copy()

print("Rows after:", len(df_model))
print("Rows removed:", len(df) - len(df_model))

print("\nRemaining missing values in history features:")
print(df_model[history_features].isnull().sum())

Rows before: 1017209
Rows after: 985989
Rows removed: 31220

Remaining missing values in history features:
SalesLag1             0
SalesLag7             0
SalesLag14            0
SalesLag28            0
SalesRollingMean7     0
SalesRollingMean28    0
SalesRollingStd7      0
SalesMomentum         0
SalesTrend            0
SalesChange7          0
dtype: int64


In [16]:
# Find categorical columns

categorical_cols = df_model.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

print("\nUnique values per categorical column:")

for col in categorical_cols:
    print(f"\n{col}:")
    print(df_model[col].value_counts(dropna=False).head(20))

Categorical columns:
['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval', 'MonthName']

Unique values per categorical column:

StateHoliday:
StateHoliday
0    956362
a     18837
b      6690
c      4100
Name: count, dtype: int64

StoreType:
StoreType
a    534771
d    303168
c    132696
b     15354
Name: count, dtype: int64

Assortment:
Assortment
a    520841
c    457106
b      8042
Name: count, dtype: int64

PromoInterval:
PromoInterval
NaN                 492799
Jan,Apr,Jul,Oct     283742
Feb,May,Aug,Nov     114956
Mar,Jun,Sept,Dec     94492
Name: count, dtype: int64

MonthName:
MonthName
Mar     103695
May     103695
Apr     100350
Jun     100350
Jul      98115
Feb      93660
Jan      72474
Aug      63550
Oct      63550
Dec      63550
Sept     61500
Nov      61500
Name: count, dtype: int64


In [17]:
# Create final modeling dataset

model_df = df_model.copy()

# Remove raw or redundant columns
columns_to_drop = [
    "PromoInterval",
    "Promo2StartDate",
    "CompetitionOpenDate",
    "MonthName"
]

model_df = model_df.drop(
    columns=columns_to_drop,
    errors="ignore"
)

# One-hot encode categorical features
categorical_features = [
    "StateHoliday",
    "StoreType",
    "Assortment"
]

model_df = pd.get_dummies(
    model_df,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("Model dataset shape:", model_df.shape)

print("\nRemaining object columns:")
print(
    model_df.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
)

print("\nFirst 10 columns:")
print(model_df.columns.tolist()[:10])

Model dataset shape: (985989, 47)

Remaining object columns:
[]

First 10 columns:
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'SchoolHoliday', 'CompetitionDistance', 'CompetitionOpenSinceMonth']


In [18]:
# Chronological train-validation split

# Sort by date
model_df = model_df.sort_values("Date").reset_index(drop=True)

# Use the last 6 weeks as validation
cutoff_date = model_df["Date"].max() - pd.Timedelta(weeks=6)

train_df = model_df[model_df["Date"] < cutoff_date].copy()
valid_df = model_df[model_df["Date"] >= cutoff_date].copy()

# Target
target = "Sales"

# Features not available or not appropriate at prediction time
drop_features = [
    "Sales",
    "Date",
    "Customers"
]

feature_cols = [
    col for col in model_df.columns
    if col not in drop_features
]

X_train = train_df[feature_cols]
y_train = train_df[target]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target]

print("Cutoff date:", cutoff_date)

print("\nTraining period:")
print(train_df["Date"].min(), "to", train_df["Date"].max())

print("\nValidation period:")
print(valid_df["Date"].min(), "to", valid_df["Date"].max())

print("\nShapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)
print("y_valid:", y_valid.shape)

print("\nNumber of model features:", len(feature_cols))

Cutoff date: 2015-06-19 00:00:00

Training period:
2013-01-29 00:00:00 to 2015-06-18 00:00:00

Validation period:
2015-06-19 00:00:00 to 2015-07-31 00:00:00

Shapes:
X_train: (938044, 44)
y_train: (938044,)
X_valid: (47945, 44)
y_valid: (47945,)

Number of model features: 44


In [19]:
# Save engineered modeling dataset

output_path = "../data/processed/model_data.pkl"

model_df.to_pickle(output_path)

print("Saved engineered dataset to:", output_path)
print("Shape:", model_df.shape)

Saved engineered dataset to: ../data/processed/model_data.pkl
Shape: (985989, 47)
